# Data Preparation

## Set Up Environment

### Install Dependencies

In [1]:
!pip install --upgrade pandas sentence-transformers pandas tqdm urllib3 Sastrawi

### Import Dependencies

In [2]:
import pandas as pd
import numpy as np
import re
import os

from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from urllib.parse import urlparse
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

c:\Users\User\miniconda3\envs\clustering_indosbert_recursive_spherical_k-means\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Model Sentence-BERT (IndoSBERT)

In [3]:
model_name = 'denaya/indoSBERT-large'

model = SentenceTransformer(model_name)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26016.86it/s]


### Load Raw Data

In [4]:
data_path = 'datasets/raw_tickets.csv'

df_raw = pd.read_csv(data_path)

df = df_raw[['DESKRIPSI']].dropna()
df.rename(columns={'DESKRIPSI': 'RAW TICKET'}, inplace=True)

df.head()

,RAW TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...


## Data Preprocessing

### Preprocessed Data for Sentence-Embedding Model

#### Replace new line special character

In [5]:
# jika terdapat lebih dari satu newline, ganti dahulu menjadi satu newline, lalu ganti newline tersebut dengan titik dan spasi
df['REPLACED NEWLINE'] = df['RAW TICKET'].apply(lambda x: re.sub(r'\n+', '. ', x))

df.head()

,RAW TICKET,REPLACED NEWLINE
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...


#### Lower Casing

In [6]:
df['LOWERCASED'] = df['REPLACED NEWLINE'].str.lower()

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASED
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...


#### Remove Klien and Placeholder 'Isu/Kendala'

In [7]:
def remove_placeholder(teks):
    if not isinstance(teks, str):
        return teks
    
    # 1a. Tiket berformat label: buang blok klien sampai sebelum label isu/kendala
    pattern_klien_berlabel = r'\b(?:(?:klien|client|clien|lien)[\s:;,\.]*|(?:dashboard trial|dashboard client|dashboard|project)\s*[:;]\s*)(.*?)(?=\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\b)'
    teks = re.sub(pattern_klien_berlabel, '', teks)
    
    # 1b. Sisa "klien: <nama>" yang mungkin masih ada (tiket tanpa label isu)
    # Hanya buang token "klien:" dan nilai langsung setelahnya sampai titik/koma berikutnya
    pattern_klien_sisa = r'\b(?:klien|client|clien|lien)[\s:;,\.]*[^.!?\n]*?(?=[.!?\n]|$)'
    teks = re.sub(pattern_klien_sisa, '', teks)
    
    # 2. Hapus label isu/kendala (tidak berubah)
    pattern_isu_detail = r'\b(?:isu|issue|kendala|request|req|keluhan|problem|kebutuhan|detail)\s*[:;]\s*'
    teks = re.sub(pattern_isu_detail, '', teks)
    
    # 3. Finalisasi
    teks = teks.strip(' ;:,-')
    teks = re.sub(r'\s+', ' ', teks)
    
    return teks

# Terapkan fungsinya
df['REMOVED PLACEHOLDER'] = df['LOWERCASED'].apply(remove_placeholder)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASED,REMOVED PLACEHOLDER
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ..."


#### Masking Nama dan Alamat Email

In [8]:
# Masking nama akun (contoh: @namaakun) dengan placeholder nama orang
df['MASKED CRED'] = df['REMOVED PLACEHOLDER'].apply(lambda x: re.sub(r'@\w+', 'nama orang', x))

# Masking email (contoh: email@domain.com) dengan placeholder email
df['MASKED CRED'] = df['MASKED CRED'].apply(lambda x: re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', 'email', x))

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASED,REMOVED PLACEHOLDER,MASKED CRED
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ..."


#### Masking Sapaan

In [9]:
# Masking sapaan dengan placeholder sapaan
sapaan_pattern = (
    r'\b('
    # Sapaan berbasis waktu (panjang & pendek)
    r'selamat pagi|selamat siang|selamat sore|selamat malam|'
    r'pagi|siang|sore|malam|'
    # Sapaan umum & informal (termasuk huruf berulang seperti halloo, haloo)
    r'hal+o+|hel+o+|hi|permisi|assalamu\'?alaikum|assalamualaikum|'
    # Sapaan daerah / Sunda
    r'punteun|punten|nuhun|'
    # Frasa permohonan bantuan
    r'mohon dibantu|mohon bantuan(?:nya)?|minta tolong|tolong dibantu|tolong bantuan(?:nya)?|'
    # Ucapan terima kasih dan penutup
    r'terima\s?kasih\s?sebelumnya|terimakasih\s?sebelumnya|'
    r'terima\s?kasih\s?banyak|terimakasih\s?banyak|'
    r'terima\s?kasih|terimakasih|makasih|'
    r'thanks|thank\s?you|guys'
    r')\b'
)
df['MASKED GREETING'] = df['MASKED CRED'].apply(lambda x: re.sub(sapaan_pattern, 'sapaan', x, flags=re.IGNORECASE))

df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASED,REMOVED PLACEHOLDER,MASKED CRED,MASKED GREETING
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...,crawlback comment. sapaan tim it sapaan bantua...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. sapaan t...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ...","dashboard loading. sapaan tim it, sapaan ada k..."


#### Remove Emoticon

In [10]:
# Hapus emotikon
def remove_emoticon(teks):
    if not isinstance(teks, str):
        return teks
    # Rentang emotikon umum (emoji, simbol, dll.)
    emoticon_pattern = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FA6F\U0001FA70-\U0001FAFF]+'
    return re.sub(emoticon_pattern, '', teks)

df['REMOVED EMOTICON'] = df['MASKED GREETING'].apply(remove_emoticon)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASED,REMOVED PLACEHOLDER,MASKED CRED,MASKED GREETING,REMOVED EMOTICON
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. sapaan tim it sapaan bantua...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","sapaan tim it, sapaan saya menemukan link dari..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. sapaan t...,data postingan instagram tidak masuk. sapaan t...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,sapaan mas nama orang dan tim info untuk siput...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ...","dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. sapaan tim it, sapaan ada k..."


#### Parse Url

In [11]:
def parse_url(teks):
    if not isinstance(teks, str):
        return teks
    
    # POLA REGEX BARU: 
    # Menangkap seluruh string URL yang valid (termasuk huruf, angka, strip, titik, dan garis miring)
    pola_url = r'https?://[\w\-\.\/\?\&\=\%]+'
    
    def ekstrak_domain(match):
        url = match.group(0)
        
        # Mencegah titik atau koma di akhir kalimat ikut terbaca sebagai bagian URL
        if url.endswith('.') or url.endswith(','):
            url = url[:-1]
            
        try:
            # urlparse akan membedah URL. 
            # Contoh: dari "https://megapolitan.kompas.com/..." kita ambil "megapolitan.kompas.com"
            netloc = urlparse(url).netloc
            
            # Pecah berdasarkan titik
            parts = netloc.split('.')
            
            # LOGIKA PENCARIAN NAMA DOMAIN UTAMA:
            # Kasus 1: Domain Indonesia 3 tingkat (contoh: news.detik.co.id -> ambil 'detik')
            if len(parts) >= 3 and parts[-2] in ['co', 'go', 'ac', 'or', 'sch', 'my']:
                domain_utama = parts[-3]
                
            # Kasus 2: Domain standar dengan/tanpa subdomain (contoh: megapolitan.kompas.com -> ambil 'kompas')
            elif len(parts) >= 2:
                domain_utama = parts[-2]
                
            # Kasus 3: Fallback jika format URL tidak biasa
            else:
                domain_utama = parts[0]
                
            # Jika domain utama tertangkap sebagai 'www', ambil kata setelahnya
            if domain_utama == 'www' and len(parts) >= 2:
                domain_utama = parts[-1]
                
            return f"tautan {domain_utama.lower()}"
            
        except Exception:
            return "tautan" # Fallback jika terjadi error parsing
            
    # Terapkan re.sub menggunakan fungsi ekstrak_domain
    teks = re.sub(pola_url, ekstrak_domain, teks)
    
    return teks

# Terapkan fungsi ke kolom yang sudah diproses sebelumnya
df['PARSED URL'] = df['REMOVED EMOTICON'].apply(parse_url)
df.head()

,RAW TICKET,REPLACED NEWLINE,LOWERCASED,REMOVED PLACEHOLDER,MASKED CRED,MASKED GREETING,REMOVED EMOTICON,PARSED URL
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,Klien: Setwapres Medsos. isu: Crawlback Commen...,klien: setwapres medsos. isu: crawlback commen...,crawlback comment. siang tim it minta tolong b...,crawlback comment. siang tim it minta tolong b...,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. sapaan tim it sapaan bantua...,crawlback comment. sapaan tim it sapaan bantua...
1,"selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","sapaan tim it, sapaan saya menemukan link dari...","sapaan tim it, sapaan saya menemukan link dari..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,Klien: BPS. Isu: Data postingan Instagram tida...,klien: bps. isu: data postingan instagram tida...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. selamat ...,data postingan instagram tidak masuk. sapaan t...,data postingan instagram tidak masuk. sapaan t...,data postingan instagram tidak masuk. sapaan t...
3,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @Dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas @dhanysybn dan tim info untuk...,selamat sore mas nama orang dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,sapaan mas nama orang dan tim info untuk siput...,sapaan mas nama orang dan tim info untuk siput...
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,Klien: Heinz . Isu: Dashboard loading. Selamat...,klien: heinz . isu: dashboard loading. selamat...,"dashboard loading. selamat sore tim it, mohon ...","dashboard loading. selamat sore tim it, mohon ...","dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. sapaan tim it, sapaan ada k...","dashboard loading. sapaan tim it, sapaan ada k..."


#### Save Results to CSV

In [12]:
# Simpan hasil akhir ke CSV baru
output_path = 'datasets/cleaned_tickets.csv'

df_result = df[['RAW TICKET', 'PARSED URL']].copy()
df_result.rename(columns={'PARSED URL': 'CLEANED TICKET'}, inplace=True)

df_result.to_csv(output_path, index=False)

### Sentence-Embedding

#### Implements Sliding Windows

In [13]:
tokenizer = model.tokenizer

def get_sliding_window_embedding(teks, model, max_length=256, stride=128):
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = 100_000
    tokens = tokenizer.encode(teks, add_special_tokens=False)
    tokenizer.model_max_length = original_max_length

    # Jika pendek, langsung encode
    if len(tokens) <= max_length - 2:
        return model.encode(teks)

    window_size = max_length - 2  # ruang untuk [CLS] dan [SEP]
    step = window_size - stride    # seberapa jauh window bergeser tiap iterasi

    chunk_embeddings = []
    start = 0

    while start < len(tokens):
        end = min(start + window_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunk_emb = model.encode(chunk_text)
        chunk_embeddings.append(chunk_emb)
        
        if end == len(tokens):  # sudah sampai akhir
            break
        start += step

    return np.mean(chunk_embeddings, axis=0)

#### Embed Sentence

In [14]:
# Mengaktifkan ekstensi pandas dari tqdm untuk memunculkan progress bar
tqdm.pandas(desc="Proses Embedding IndoSBERT")

# 1. Mencegah Error: Pastikan tidak ada data kosong (NaN) akibat proses pembersihan sebelumnya
df_result['CLEANED TICKET'] = df_result['CLEANED TICKET'].fillna("").astype(str)

# 2. Ekstraksi Embedding dengan Sliding Window
df_result['EMBEDDING'] = df_result['CLEANED TICKET'].progress_apply(
    lambda teks: get_sliding_window_embedding(
        teks=teks, 
        model=model, 
        max_length=256, 
        stride=128
    )
)

# Menampilkan 5 baris pertama untuk memastikan kolom EMBEDDING sudah terbentuk
df_result.head()

Proses Embedding IndoSBERT: 100%|██████████| 1639/1639 [04:32<00:00,  6.02it/s]


,RAW TICKET,CLEANED TICKET,EMBEDDING
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,crawlback comment. sapaan tim it sapaan bantua...,"[0.00083771703, -0.58585614, 0.064690515, 0.08..."
1,"selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","[0.02750907, 0.037827827, -0.5283966, -0.13133..."
2,Klien: BPS\nIsu: Data postingan Instagram tida...,data postingan instagram tidak masuk. sapaan t...,"[-0.17697816, -0.045605347, -0.3471804, 0.4249..."
3,selamat sore mas @Dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,"[0.08603829, -0.017807081, 0.016980363, 0.3204..."
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,"dashboard loading. sapaan tim it, sapaan ada k...","[0.14301413, 0.028007613, 0.030197145, 0.24381..."


### Preprocessed Data for Word Cloud

In [15]:
keywords_to_keep = {
    'belum', 'tidak', 'kurang', 'bisa', 'semua', 'atas', 'bawah', 
    'masuk', 'keluar', 'hilang', 'kosong', 'gagal', 'salah'
}

khas_tiket_noise = {
    'mas', 'mba', 'om', 'tim', 'it', 'mohon', 'dibantu', 'punteun', 'tolong', 
    'cek', 'bantu', 'halo', 'min', 'admin', 'dari', 'ke', 'perihal', 'tautan',
    'siang', 'pagi', 'sore', 'malam', 'selamat', 'terima', 'kasih', 'sebelumnya',
    'ya', 'yaa', 'kah', 'lah', 'pun', 'terimakasih', 'salam', 'sapaan', 'nama', 'orang',
    'email', 'domain', 'com', 'kak', 'guys', 'klien', 'client', 'user', 'bantuannya', 
    'makasih', 'nuhun', 'punten', 'maaf', 'izin', 'tanya', 'informasi', 'info', 'link', 
    'url', 'kolom', 'kendala', 'isu', 'issue', 'muncul', 'sesuai', 'terkait', 'dicek', 
    'update', 'tanggal', 'jam', 'periode', 'nya', 'yg', 'ga', 'gak', 'gaada', 'gabisa', 
    'udah', 'udh', 'buat', 'kalo', 'kalau', 'tapi', 'tetap', 'ttp', 'sih', 'kok', 'kan', 
    'dong', 'deh', 'mah', 'aja', 'terus', 'data', 'mati'
}

factory = StopWordRemoverFactory()
stopword_indonesia = factory.get_stop_words()

stopwords = set(stopword_indonesia) - keywords_to_keep
stopwords.update(khas_tiket_noise)

In [16]:
def normalize_token(word):
    cleaned = re.sub(r'^\W+|\W+$', '', word.lower())
    return cleaned


df_result['WORDCLOUD TICKET'] = df_result['CLEANED TICKET'].apply(
    lambda x: ' '.join(
        [word for word in x.split() if normalize_token(word) and normalize_token(word) not in stopwords]
    )
)


df_result.head()

,RAW TICKET,CLEANED TICKET,EMBEDDING,WORDCLOUD TICKET
0,Klien: Setwapres Medsos\nisu: Crawlback Commen...,crawlback comment. sapaan tim it sapaan bantua...,"[0.00083771703, -0.58585614, 0.064690515, 0.08...",crawlback comment. bantuan crawlback comment m...
1,"selamat pagi tim it, mohon bantuannya saya men...","sapaan tim it, sapaan saya menemukan link dari...","[0.02750907, 0.037827827, -0.5283966, -0.13133...",menemukan megapolitan.kompas.com tidak masuk d...
2,Klien: BPS\nIsu: Data postingan Instagram tida...,data postingan instagram tidak masuk. sapaan t...,"[-0.17697816, -0.045605347, -0.3471804, 0.4249...",postingan instagram tidak masuk. apa bikin pos...
3,selamat sore mas @Dhanysybn dan tim info untuk...,sapaan mas nama orang dan tim info untuk siput...,"[0.08603829, -0.017807081, 0.016980363, 0.3204...",siputri tidak bisa akses minta yaaa.
4,Klien: Heinz \nIsu: Dashboard loading\n\nSelam...,"dashboard loading. sapaan tim it, sapaan ada k...","[0.14301413, 0.028007613, 0.030197145, 0.24381...",dashboard loading. keluhan pihak heinz dashboa...


#### Save to Pickle

In [ ]:
output_path = 'datasets/tickets.pkl'

df_result.to_pickle(output_path)

: 